# Amazon Entity Resolution - Kaggle training (V7)

This notebook clones the project code, installs the runtime packages, and trains LightGBM using all supplied training entities. It samples only easy negative candidate pairs; it does not subset the source tables.

## Before running

1. Import this notebook into Kaggle.
2. Prefer Kaggle **Add Input** and attach the official competition data. Accept the competition rules if prompted. The expected layout has `train/` and `test/` as sibling folders.
3. If using the private GitHub LFS archive instead, store a GitHub token in Kaggle Secrets as `GITHUB_TOKEN`. Private LFS downloads may require a classic personal access token with `repo` scope; a fine-grained token may clone the source but be denied by Git LFS. Never put the token in a code cell.
4. Select a CPU runtime with enough free working disk space. GPU is not required. Run cells in order; outputs go under `/kaggle/working/v7` and `/kaggle/working/output`.

The input finder searches recursively under `/kaggle/input/` for the required TSVs. If it finds no attached data, the notebook attempts the private LFS archive and prints the Kaggle input roots to help diagnose the mount path.


In [ ]:
from pathlib import Path
import base64, os, shutil, subprocess, sys

REPO_URL = 'https://github.com/rohitkumar8279/AMAZON-ML-CHALLENGE.git'
REPO_DIR = Path('/kaggle/working/AMAZON-ML-CHALLENGE')
GITHUB_USERNAME = 'rohitkumar8279'  # set this to the account that created GITHUB_TOKEN
try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Add Kaggle Secret GITHUB_TOKEN with a GitHub token that can read this private repository.') from exc
auth_env = os.environ.copy()
auth_env['GIT_CONFIG_COUNT'] = '1'
auth_env['GIT_CONFIG_KEY_0'] = 'http.https://github.com/.extraheader'
# GitHub PAT over HTTPS uses the account username and token as the password.
basic = base64.b64encode((GITHUB_USERNAME + ':' + github_token).encode()).decode()
auth_env['GIT_CONFIG_VALUE_0'] = 'AUTHORIZATION: basic ' + basic
auth_env['GIT_LFS_SKIP_SMUDGE'] = '1'  # avoid downloading the 1.09 GB archive unless needed
probe = subprocess.run(['git', 'ls-remote', REPO_URL, 'HEAD'], env=auth_env, capture_output=True, text=True)
if probe.returncode:
    raise RuntimeError('GitHub denied repository read access. Replace GITHUB_TOKEN with an unexpired token for the correct account. Fine-grained token: select rohitkumar8279/AMAZON-ML-CHALLENGE and grant Contents: read (Metadata: read); classic token: grant repo access. Do not paste the token into notebook code.')
# Clear an incomplete directory left by an earlier failed clone.
if REPO_DIR.exists() and not (REPO_DIR / '.git').is_dir():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True, env=auth_env)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True, env=auth_env)
print('Project:', REPO_DIR)
print('Training script:', REPO_DIR / 'code/business_entity_resolution/src/train_v7.py')


In [ ]:
import subprocess, sys

# Kaggle already provides NumPy and SciPy. Install only project runtime packages,
# without asking pip to replace Kaggle's shared scientific stack.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                'lightgbm==4.7.0', 'rapidfuzz==3.14.3', 'Unidecode==1.4.0'], check=True)
import numpy, scipy, lightgbm, rapidfuzz, unidecode
print('Runtime ready:', 'numpy', numpy.__version__, '| lightgbm', lightgbm.__version__)


In [ ]:
from pathlib import Path, PurePosixPath
import subprocess, zipfile

INPUT_ROOT = Path('/kaggle/input')
TRAIN_REQUIRED = {'train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv', 'train_ground_truth.tsv'}
TEST_REQUIRED = {'test_source1.tsv', 'test_source2.tsv', 'test_source3.tsv'}
def find_attached_dataset():
    found = []
    if INPUT_ROOT.exists():
        for source1 in INPUT_ROOT.rglob('train_source1.tsv'):
            train_dir = source1.parent
            test_dir = train_dir.parent / 'test'
            if TRAIN_REQUIRED.issubset({p.name for p in train_dir.iterdir() if p.is_file()}) and test_dir.is_dir() and TEST_REQUIRED.issubset({p.name for p in test_dir.iterdir() if p.is_file()}):
                found.append((train_dir, test_dir))
    return found
def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            rel = PurePosixPath(member.filename)
            if rel.is_absolute() or '..' in rel.parts:
                raise RuntimeError(f'Unsafe path in attached archive: {member.filename}')
            target = (destination / Path(*rel.parts)).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'Archive member escapes extraction folder: {member.filename}')
        zf.extractall(destination)
matches = find_attached_dataset()
print('Kaggle input roots:', [str(p) for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else 'missing /kaggle/input')
# Kaggle may mount the uploaded student-resource ZIP without unpacking it. Extract it locally first.
if not matches and INPUT_ROOT.exists():
    archives = [p for p in INPUT_ROOT.rglob('*.zip') if 'student_resource' in p.name.lower()]
    for input_archive in archives:
        with zipfile.ZipFile(input_archive) as zf:
            members = [n.replace('\\', '/') for n in zf.namelist()]
        required_basenames = TRAIN_REQUIRED | TEST_REQUIRED
        if not required_basenames.issubset({Path(n).name for n in members}):
            continue
        extract_root = Path('/kaggle/working/challenge_data_from_input')
        safe_extract(input_archive, extract_root)
        print('Extracted Kaggle input archive:', input_archive)
        matches = find_attached_dataset()
        if matches:
            break
if not matches:
    archive = REPO_DIR / '6ab10eb3b23ba_student_resource.zip'
    download_env = auth_env.copy()
    download_env.pop('GIT_LFS_SKIP_SMUDGE', None)
    try:
        subprocess.run(['git', '-C', str(REPO_DIR), 'lfs', 'pull', '--include=6ab10eb3b23ba_student_resource.zip'], check=True, env=download_env)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('No extracted dataset or student_resource ZIP was found under /kaggle/input. Attach the official data with Kaggle Add Input. If using the private GitHub LFS archive, replace GITHUB_TOKEN with a token Git LFS accepts (a classic PAT with repo scope may be required).') from exc
    if not archive.is_file() or archive.stat().st_size < 1_000_000_000:
        raise RuntimeError('Git LFS archive was not downloaded. Attach the official challenge dataset in Kaggle Add Input or verify GitHub LFS token access.')
    extract_root = Path('/kaggle/working/challenge_data_from_github')
    safe_extract(archive, extract_root)
    matches = find_attached_dataset()
if len(matches) != 1:
    raise RuntimeError(f'Expected exactly one complete challenge dataset; found {len(matches)}. Expected train_source1.tsv, train_source2.tsv, train_source3.tsv, train_ground_truth.tsv in train/, and the three test_source*.tsv files in sibling test/. Kaggle roots: {[str(p) for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else []}')
TRAIN_DIR, TEST_DIR = matches[0]
print('Train:', TRAIN_DIR)
print('Test: ', TEST_DIR)
for p in sorted(TRAIN_DIR.iterdir()):
    if p.is_file(): print(f'{p.name}: {p.stat().st_size / 1e9:.2f} GB')


## Train and select thresholds

The default candidate cap is 64 per target source. This bounds expensive pair features while preserving the full entity tables. The saved validation report includes candidate recall and the candidate-only oracle F0.5; those values diagnose blocking losses separately from model errors.

In [ ]:
import os
WORK_DIR = Path('/kaggle/working/v7')
WORK_DIR.mkdir(parents=True, exist_ok=True)
MAX_CANDIDATES_PER_SOURCE = 64
HARD_NEGATIVES_PER_SOURCE = 8
cmd = [sys.executable, str(REPO_DIR / 'code/business_entity_resolution/src/train_v7.py'),
       '--train-dir', str(TRAIN_DIR), '--work-dir', str(WORK_DIR),
       '--max-candidates-per-source', str(MAX_CANDIDATES_PER_SOURCE),
       '--hard-negatives-per-source', str(HARD_NEGATIVES_PER_SOURCE)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)
report_path = WORK_DIR / 'validation_report_v7.json'
print(report_path.read_text() if report_path.exists() else 'Training completed; report not found at expected path.')


## Full test inference and submission files

Run after successful training. This processes the complete test tables using the same candidate cap and saved source thresholds. Then download the two TSVs from the Kaggle notebook Output panel. Validate them with the included challenge validator before submission.

In [ ]:
OUT_DIR = Path('/kaggle/working/output')
OUT_DIR.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(REPO_DIR / 'code/business_entity_resolution/src/predict_v7.py'),
       '--test-dir', str(TEST_DIR), '--model-dir', str(WORK_DIR),
       '--output-dir', str(OUT_DIR)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)
for name in ('matching_results.tsv', 'candidate_pairs.tsv'):
    path = OUT_DIR / name
    print(name, f'{path.stat().st_size / 1e6:.1f} MB' if path.exists() else 'MISSING')


In [ ]:
validator = REPO_DIR / 'utils/validate_submission.py'
cmd = [sys.executable, str(validator), '--matching', str(OUT_DIR / 'matching_results.tsv'),
       '--candidate', str(OUT_DIR / 'candidate_pairs.tsv'), '--test-dir', str(TEST_DIR)]
subprocess.run(cmd, cwd=REPO_DIR, check=True)
